# Marker-Pair Proximity Colocalization Analysis

Explore per-cell spatial colocalization using `pg.proximity()` join-count z-scores.

**Strategy:**
1. Load proximity data (join_count_z per cell per marker pair)
2. Select top 1000 most variable pairs across all cells
3. **Within each cell system**: compare 6h vs 48h, Mock vs Blinatumomab
4. **Between cell systems**: compare in each of the 4 conditions (Mock_6h, Blina_6h, Mock_48h, Blina_48h)

Each comparison produces:
- Heatmap of mean z-score in group A and group B
- Heatmap of log2 fold-change (or difference) between groups
- Volcano plot (-log10 p vs effect size)
- Heatmaps of the top 2 most differentially colocalized pairs across cells

In [ ]:
import sys
sys.path.insert(0, '/home/projects/nyosef/zvise/PixelGen/PixelGen')

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Global figure style ---
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "font.family": "sans-serif",
    "figure.dpi": 120,
})

CACHE_DIR = Path("/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache")

TOP_N_PAIRS = 1000  # Number of most variable pairs to analyze

# Color palettes
COND_PALETTE = {"Mock": "#4c72b0", "Blinatumomab": "#dd8452"}
TIME_PALETTE = {"6h": "#55a868", "48h": "#c44e52"}
SYSTEM_PALETTE = {
    "healthy B + healthy T": "#4878d0",
    "NALM-6 + healthy T": "#ee854a",
    "patient B + patient T": "#6acc65",
    "NALM-6 + patient T": "#d65f5f",
}

ISOTYPE_CONTROLS = {"mIgG1", "mIgG2a", "mIgG2b"}

## 1. Load Data & Build Per-Cell Colocalization Matrix

In [ ]:
# Load annotated adata with cell type labels and spatial proximity features
ANNOTATED_CACHE = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'
adata = sc.read_h5ad(ANNOTATED_CACHE)

# Filter to CD8 T cells only
adata = adata[adata.obs["cell_type"] == "CD8"].copy()

# Standardise column names
if "system" not in adata.obs.columns and "cell_system" in adata.obs.columns:
    adata.obs["system"] = adata.obs["cell_system"]

# Derive cond_time if not present
if "cond_time" not in adata.obs.columns:
    adata.obs["cond_time"] = adata.obs["condition"].astype(str) + "_" + adata.obs["time"].astype(str)

print(f"adata (CD8 only): {adata.shape}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"spatial_raw shape: {adata.obsm['spatial_raw'].shape}")
print()
print(adata.obs.groupby(["system", "cond_time"]).size().unstack(fill_value=0))

In [ ]:
# Use the precomputed spatial_raw (join_count_z) from obsm
# Filter out isotype pairs, select top N most variable
full_z = adata.obsm["spatial_raw"]

if isinstance(full_z, pd.DataFrame):
    # Drop pairs involving isotype controls
    iso_pairs = [c for c in full_z.columns
                 if any(iso in c for iso in ISOTYPE_CONTROLS)]
    full_z = full_z.drop(columns=iso_pairs, errors="ignore")
    print(f"Dropped {len(iso_pairs)} isotype pairs")

    # Select top N by variance
    pair_var = full_z.var(axis=0).sort_values(ascending=False)
    top_pairs = pair_var.head(TOP_N_PAIRS).index.tolist()
    cell_pair_z = full_z[top_pairs].copy()
else:
    raise ValueError("spatial_raw should be a DataFrame with pair names as columns")

print(f"Full spatial_raw matrix: {full_z.shape}")
print(f"Selected top {TOP_N_PAIRS} variable pairs: {cell_pair_z.shape}")
print(f"Variance range: [{pair_var.iloc[min(TOP_N_PAIRS-1, len(pair_var)-1)]:.2f}, {pair_var.iloc[0]:.2f}]")

In [ ]:
# Metadata is already in adata.obs
meta_cols = ["sample", "time", "condition", "system", "cond_time"]
# Add cell_type if available
if "cell_type" in adata.obs.columns:
    meta_cols.append("cell_type")
cell_meta = adata.obs[meta_cols].copy()
cell_pair_z.index = cell_meta.index  # ensure aligned

# Detect pair separator used in column names (/ or :)
sample_col = cell_pair_z.columns[0]
PAIR_SEP = "/" if "/" in sample_col else ":"
print(f"Pair separator: '{PAIR_SEP}'  (example: {sample_col})")
print(f"Cells with metadata: {len(cell_meta)}")
print(cell_meta["system"].value_counts())

## 2. Helper Functions

In [ ]:
def differential_proximity(z_mat, meta, group_col, group_a, group_b, min_cells=10):
    """
    Mann-Whitney U test per pair between two groups.

    Returns DataFrame with columns:
        pair, mean_a, mean_b, diff (b - a), log2fc, pval, pval_adj, neg_log10p
    """
    mask_a = meta[group_col] == group_a
    mask_b = meta[group_col] == group_b
    cells_a = mask_a.sum()
    cells_b = mask_b.sum()

    if cells_a < min_cells or cells_b < min_cells:
        print(f"  Skipping: {group_a} ({cells_a} cells) vs {group_b} ({cells_b} cells)")
        return None

    mat_a = z_mat.loc[mask_a]
    mat_b = z_mat.loc[mask_b]

    results = []
    for pair in z_mat.columns:
        va = mat_a[pair].values
        vb = mat_b[pair].values
        mean_a = np.mean(va)
        mean_b = np.mean(vb)
        diff = mean_b - mean_a

        # Avoid log2(0) issues: use pseudo-shifted means for log2fc
        pseudo = 0.01
        log2fc = np.log2((abs(mean_b) + pseudo) / (abs(mean_a) + pseudo))
        if (mean_b - mean_a) < 0:
            log2fc = -abs(log2fc)

        try:
            _, pval = mannwhitneyu(va, vb, alternative="two-sided")
        except ValueError:
            pval = 1.0

        results.append({
            "pair": pair,
            "mean_a": mean_a,
            "mean_b": mean_b,
            "diff": diff,
            "log2fc": log2fc,
            "pval": pval,
        })

    df = pd.DataFrame(results)
    _, df["pval_adj"], _, _ = multipletests(df["pval"], method="fdr_bh")
    df["neg_log10p"] = -np.log10(df["pval"].clip(lower=1e-300))
    df = df.sort_values("pval")
    return df


def split_pair(pair_str):
    """Split 'A/B' or 'A:B' into (A, B)."""
    return tuple(pair_str.split(PAIR_SEP))

In [ ]:
def plot_comparison_suite(diff_df, z_mat, meta, group_col, group_a, group_b,
                         title_prefix="", n_top_pairs=2, full_z_mat=None):
    """
    Full comparison visualization:
      1. Clustered heatmap of mean z-score in group A (ALL markers from full_z_mat)
      2. Clustered heatmap of mean z-score in group B
      3. Clustered heatmap of difference (B - A)
      4. Volcano plot (x = log2FC, labels = top 10 by |log2FC|)
      5. Top N pairs by |log2FC| — violin + strip plots comparing distributions

    full_z_mat: the complete z-score matrix (all non-isotype pairs) used to
                build marker×marker heatmaps so ALL proteins are shown.
                Falls back to z_mat if not provided.
    """
    if diff_df is None or len(diff_df) == 0:
        return

    mask_a = meta[group_col] == group_a
    mask_b = meta[group_col] == group_b

    # --- 1-3: Group heatmaps using ALL pairs from full_z_mat ---
    heatmap_z = full_z_mat if full_z_mat is not None else z_mat
    # Subset to cells in this comparison
    heatmap_z_sub = heatmap_z.loc[mask_a | mask_b]
    all_pairs_list = heatmap_z_sub.columns.tolist()

    markers_in_all = set()
    for p in all_pairs_list:
        a, b = split_pair(p)
        markers_in_all.add(a)
        markers_in_all.add(b)
    markers_sorted = sorted(markers_in_all)

    def build_mean_matrix(mask):
        mat = pd.DataFrame(0.0, index=markers_sorted, columns=markers_sorted)
        sub = heatmap_z.loc[mask, all_pairs_list].mean(axis=0)
        for pair_name, val in sub.items():
            a, b = split_pair(pair_name)
            mat.loc[a, b] = val
            mat.loc[b, a] = val
        return mat

    mat_a = build_mean_matrix(mask_a)
    mat_b = build_mean_matrix(mask_b)
    mat_diff = mat_b - mat_a

    n_markers = len(markers_sorted)
    fig_size = max(10, n_markers * 0.18)
    vmax_ab = max(mat_a.abs().max().max(), mat_b.abs().max().max())
    vmax_diff = mat_diff.abs().max().max()

    # Use clustermap for hierarchical clustering
    from scipy.cluster.hierarchy import linkage
    from scipy.spatial.distance import squareform

    # Compute linkage once from the difference matrix
    dist_mat = mat_diff.abs()
    dist_for_clust = dist_mat.max().max() - dist_mat + 1e-10
    np.fill_diagonal(dist_for_clust.values, 0)
    condensed = squareform(dist_for_clust.values, checks=False)
    row_linkage = linkage(condensed, method="ward")

    for mat, label, cmap, vm in [
        (mat_a, f"{group_a} (mean z)", "RdBu_r", vmax_ab),
        (mat_b, f"{group_b} (mean z)", "RdBu_r", vmax_ab),
        (mat_diff, f"Diff ({group_b} - {group_a})", "coolwarm", vmax_diff),
    ]:
        g = sns.clustermap(mat, cmap=cmap, center=0, vmin=-vm, vmax=vm,
                           row_linkage=row_linkage, col_linkage=row_linkage,
                           figsize=(fig_size, fig_size), linewidths=0,
                           xticklabels=True, yticklabels=True,
                           cbar_kws={"shrink": 0.4, "label": label},
                           dendrogram_ratio=0.08, cbar_pos=(0.02, 0.82, 0.03, 0.15))
        g.ax_heatmap.tick_params(axis="both", labelsize=max(4, min(7, 180 // n_markers)))
        g.fig.suptitle(f"{title_prefix}: {label} (all {n_markers} markers)", fontsize=12, y=1.01)
        plt.show()

    # --- Precompute significant pairs ranked by |log2FC| (used for volcano labels + violins) ---
    sig_mask = diff_df["pval_adj"] < 0.05
    sig_by_lfc = diff_df[sig_mask].assign(abs_log2fc=lambda d: d["log2fc"].abs()).sort_values("abs_log2fc", ascending=False)

    # --- 4: Volcano plot ---
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(diff_df.loc[~sig_mask, "log2fc"], diff_df.loc[~sig_mask, "neg_log10p"],
               s=8, alpha=0.4, color="grey", label="NS")
    ax.scatter(diff_df.loc[sig_mask, "log2fc"], diff_df.loc[sig_mask, "neg_log10p"],
               s=12, alpha=0.6, color="#c44e52", label=f"FDR < 0.05 (n={sig_mask.sum()})")

    # Label top 10 significant pairs by highest absolute log2FC
    for _, row in sig_by_lfc.head(10).iterrows():
        ax.annotate(row["pair"], (row["log2fc"], row["neg_log10p"]),
                    fontsize=6, alpha=0.8, ha="center",
                    xytext=(0, 5), textcoords="offset points")

    ax.axhline(-np.log10(0.05), ls="--", color="grey", lw=0.8, alpha=0.5)
    ax.set_xlabel(f"log2 fold-change ({group_b} / {group_a})")
    ax.set_ylabel("-log10(p-value)")
    ax.set_title(f"{title_prefix}: Volcano plot")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    # --- 5: Top N pairs by |log2FC| — violin + strip distribution plots ---
    top_pair_names = sig_by_lfc.head(n_top_pairs)["pair"].tolist()
    if not top_pair_names:
        return

    fig, axes_row = plt.subplots(1, len(top_pair_names),
                                  figsize=(6 * len(top_pair_names), 5),
                                  squeeze=False)
    palette = {group_a: "#4878d0", group_b: "#ee854a"}

    for j, pair_name in enumerate(top_pair_names):
        ax = axes_row[0, j]
        plot_data = pd.DataFrame({
            "join_count_z": z_mat.loc[mask_a | mask_b, pair_name].values,
            "group": meta.loc[mask_a | mask_b, group_col].values,
        })
        sns.violinplot(data=plot_data, x="group", y="join_count_z",
                       order=[group_a, group_b], palette=palette,
                       inner="box", cut=0, ax=ax, linewidth=0.8)
        sns.stripplot(data=plot_data, x="group", y="join_count_z",
                      order=[group_a, group_b], color="k",
                      alpha=0.08, size=1.5, jitter=True, ax=ax)

        row_info = diff_df[diff_df["pair"] == pair_name].iloc[0]
        ax.set_title(f"{pair_name}\nlog2FC={row_info['log2fc']:.2f}, FDR={row_info['pval_adj']:.2e}",
                     fontsize=10)
        ax.set_xlabel("")
        ax.set_ylabel("join_count_z" if j == 0 else "")

    fig.suptitle(f"{title_prefix}: Top {n_top_pairs} pairs by |log2FC|", fontsize=12, y=1.03)
    plt.tight_layout()
    plt.show()


def significance_star(p):
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return "ns"

## 3. Within-System Comparisons

For each cell system, compare:
- **6h vs 48h** (pooling Mock + Blinatumomab)
- **Mock vs Blinatumomab** (pooling 6h + 48h)

Each comparison: group heatmaps, difference heatmap, volcano, top-2-pair cell-level plots.

In [ ]:
# =============================================================================
# Within-system: 6h vs 48h and Mock vs Blinatumomab
# =============================================================================

systems = sorted(cell_meta["system"].unique())
within_results = {}

for system in systems:
    sys_mask = cell_meta["system"] == system
    sys_z = cell_pair_z.loc[sys_mask]
    sys_meta = cell_meta.loc[sys_mask]
    sys_full_z = full_z.loc[sys_mask]  # all pairs for heatmaps

    print(f"\n{'='*70}")
    print(f"SYSTEM: {system}  (n={sys_mask.sum()} cells)")
    print(f"{'='*70}")

    # --- 6h vs 48h ---
    print(f"\n--- 6h vs 48h ---")
    diff_time = differential_proximity(sys_z, sys_meta, "time", "6h", "48h")
    if diff_time is not None:
        n_sig = (diff_time["pval_adj"] < 0.05).sum()
        print(f"  Significant pairs (FDR<0.05): {n_sig}/{len(diff_time)}")
        within_results[(system, "6h_vs_48h")] = diff_time
        plot_comparison_suite(
            diff_time, sys_z, sys_meta, "time", "6h", "48h",
            title_prefix=f"{system} | 6h vs 48h",
            full_z_mat=sys_full_z
        )

    # --- Mock vs Blinatumomab ---
    print(f"\n--- Mock vs Blinatumomab ---")
    diff_cond = differential_proximity(sys_z, sys_meta, "condition", "Mock", "Blinatumomab")
    if diff_cond is not None:
        n_sig = (diff_cond["pval_adj"] < 0.05).sum()
        print(f"  Significant pairs (FDR<0.05): {n_sig}/{len(diff_cond)}")
        within_results[(system, "Mock_vs_Blina")] = diff_cond
        plot_comparison_suite(
            diff_cond, sys_z, sys_meta, "condition", "Mock", "Blinatumomab",
            title_prefix=f"{system} | Mock vs Blina",
            full_z_mat=sys_full_z
        )

## 4. Within-System Summary: Number of Significant Pairs per Comparison

In [ ]:
# Summary table of within-system results
summary_rows = []
for (system, comparison), df in within_results.items():
    n_sig = (df["pval_adj"] < 0.05).sum()
    top_pair = df.iloc[0]["pair"]
    top_fdr = df.iloc[0]["pval_adj"]
    top_diff = df.iloc[0]["diff"]
    summary_rows.append({
        "system": system, "comparison": comparison,
        "n_sig_FDR05": n_sig, "n_tested": len(df),
        "top_pair": top_pair, "top_FDR": f"{top_fdr:.2e}", "top_diff": f"{top_diff:.3f}",
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

## 5. Cross-System Comparisons

Compare each pair of cell systems within each of the 4 conditions (Mock_6h, Blina_6h, Mock_48h, Blina_48h).

In [ ]:
# =============================================================================
# Cross-system comparisons within each condition
# =============================================================================

from itertools import combinations

cond_times = ["Mock_6h", "Blinatumomab_6h", "Mock_48h", "Blinatumomab_48h"]
cross_results = {}

for ct in cond_times:
    ct_mask = cell_meta["cond_time"] == ct
    ct_systems = cell_meta.loc[ct_mask, "system"].unique()

    if len(ct_systems) < 2:
        print(f"\n{ct}: Only {len(ct_systems)} system(s), skipping cross-system comparison")
        continue

    print(f"\n{'='*70}")
    print(f"CONDITION: {ct}")
    print(f"{'='*70}")

    for sys_a, sys_b in combinations(sorted(ct_systems), 2):
        # Subset to this condition
        sub_mask = ct_mask & cell_meta["system"].isin([sys_a, sys_b])
        sub_z = cell_pair_z.loc[sub_mask]
        sub_meta = cell_meta.loc[sub_mask]
        sub_full_z = full_z.loc[sub_mask]  # all pairs for heatmaps

        print(f"\n  {sys_a} vs {sys_b}")
        diff = differential_proximity(sub_z, sub_meta, "system", sys_a, sys_b)
        if diff is not None:
            n_sig = (diff["pval_adj"] < 0.05).sum()
            print(f"    Significant pairs (FDR<0.05): {n_sig}/{len(diff)}")
            cross_results[(ct, sys_a, sys_b)] = diff
            plot_comparison_suite(
                diff, sub_z, sub_meta, "system", sys_a, sys_b,
                title_prefix=f"{ct} | {sys_a} vs {sys_b}",
                full_z_mat=sub_full_z
            )

## 6. Cross-System Summary

In [ ]:
# Summary table + heatmap of number of significant pairs across all cross-system comparisons
cross_summary = []
for (ct, sys_a, sys_b), df in cross_results.items():
    n_sig = (df["pval_adj"] < 0.05).sum()
    cross_summary.append({
        "cond_time": ct, "system_A": sys_a, "system_B": sys_b,
        "n_sig_FDR05": n_sig, "n_tested": len(df),
        "top_pair": df.iloc[0]["pair"],
        "top_FDR": f"{df.iloc[0]['pval_adj']:.2e}",
    })
cross_summary_df = pd.DataFrame(cross_summary)
print(cross_summary_df.to_string(index=False))

# Heatmap: n_sig per (system-pair × cond_time)
if len(cross_summary_df) > 0:
    cross_summary_df["pair_label"] = cross_summary_df["system_A"] + "\nvs\n" + cross_summary_df["system_B"]
    pivot = cross_summary_df.pivot_table(
        index="pair_label", columns="cond_time", values="n_sig_FDR05", fill_value=0
    )
    ct_cols = [c for c in cond_times if c in pivot.columns]
    pivot = pivot[ct_cols]

    fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.8)))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="YlOrRd", ax=ax,
                linewidths=0.5, cbar_kws={"label": "# significant pairs (FDR<0.05)"})
    ax.set_title("Cross-system: significant colocalization differences per condition", fontsize=12)
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

## 7. Global Overview: Top Recurring Differential Pairs

Which marker pairs show up as significant across multiple comparisons?

In [ ]:
# Collect all significant pairs across all comparisons
all_sig_pairs = []

for key, df in {**within_results, **cross_results}.items():
    sig = df[df["pval_adj"] < 0.05]
    for _, row in sig.iterrows():
        all_sig_pairs.append({"comparison": str(key), "pair": row["pair"],
                              "diff": row["diff"], "pval_adj": row["pval_adj"]})

all_sig_df = pd.DataFrame(all_sig_pairs)
if len(all_sig_df) > 0:
    # Count how many comparisons each pair is significant in
    pair_counts = all_sig_df.groupby("pair").agg(
        n_comparisons=("comparison", "nunique"),
        mean_abs_diff=("diff", lambda x: np.mean(np.abs(x))),
        min_fdr=("pval_adj", "min"),
    ).sort_values("n_comparisons", ascending=False)

    print(f"Total unique significant pairs: {len(pair_counts)}")
    print(f"\nTop 20 most recurrent pairs:")
    print(pair_counts.head(20).to_string())

    # Bar plot of top 20
    top20 = pair_counts.head(20)
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(range(len(top20)), top20["n_comparisons"], color="#4878d0", alpha=0.8)
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20.index, fontsize=8)
    ax.set_xlabel("# comparisons where significant (FDR < 0.05)")
    ax.set_title("Most recurrently differential marker pairs across all comparisons")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No significant pairs found across any comparison.")